In [ ]:
#####################################
# This creates "track figure"
#####################################
import sys
# import modules in other directories
sys.path.append('../Utils/')

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature



import trax_util as trx
import ibtracs as IBT
import parse_tempest_file as TM

import importlib
import PlotUtil as Pu

importlib.reload( TM )
importlib.reload( Pu )
importlib.reload( trx )
importlib.reload( IBT )


In [ ]:

f='/glade/work/stepheba/post/tempestextremes/high-res-l93-pmf_zmconv1200/trajectories.txt.high-res-ne120-l93-zmconv1200'
f='/glade/work/stepheba/post/tempestextremes/amip_1yr/trajectories.txt.amip_ne120'
f='/glade/work/stepheba/post/tempestextremes/xic_x02_ne240_pg3/trajectories.txt.julio_xic_x02'

camexp=f

columns = ['cell_id','lon','lat','slp','wind','xxx1','xxx2','year','month','day','hour',]
columns = ['cell_id','lon','lat','slp','wind','xxx1','year','month','day','hour',]

#dS = TM.parse_storm_file( f )
power_wind=(1./2.)**(0.11)
pdx = TM.readtrx( f ,columns=columns , power_wind=power_wind )

ibx_basins = IBT.basin_outlines(Mediterranean=False,xNEPacific=True)
pdx_basins = trx.find_basin(pdx,ibx_basins)
pdx.basin=pdx_basins


In [ ]:
#f=trx.rcp85fname(ens=1,sst='sst3')
#f=trx.pdfname(ens=1)
power_wind=(1./6.)**(0.11)

ibx=IBT.readtrx()



In [ ]:
#Maria
yook = np.where( ( ibx.year[:,0] >= 2017 )&( ibx.year[:,0] <= 2017 )& (ibx.month[:,0] == 9) & (ibx.day[:,0] >1 ) )
#Aug-Sep 2020
yook = np.where( ( ibx.year[:,0] == 2020 ) &  (ibx.month[:,0] >= 7) &  (ibx.month[:,0] <= 9) )
#2000-2010
yook = np.where( ( ibx.year[:,0] >= 2000 ) & ( ibx.year[:,0] <= 2010 ) )
#6/2004-12/2004
yook = np.where( ( ( ibx.year[:,0] == 2004 ) & (ibx.month[:,0]>=6) )  | ( ( ibx.year[:,0] == 2005 ) & (ibx.month[:,0]==1) )  )
len(yook[0])

In [ ]:
from matplotlib.legend_handler import HandlerPathCollection

central_longitude=180.

#projPC = ccrs.Robinson(central_longitude=180.)
projPC = ccrs.PlateCarree(central_longitude=central_longitude) #180.)
data_crs = ccrs.PlateCarree()

basi = IBT.basin_outlines(Mediterranean=False,xNEPacific=True)
ipan=-1
"""
Add axes method 
ax = fig.add_axes([xmin , ymin , dx , dy , ])
"""
# Color mapping for categories
categories = np.arange(-2, 6)  # -2 (sub-category storms) to 7 (Cat5)
colors = ['gray', 'blue', 'cyan', 'green', 'yellow', 'orange', 'purple', 'red']  # Red for Cat5+


scale_fac = 1e-4 * 1.944**2 
denslv=21 #np.linspace(0,15,num=16)
#denslv=np.linspace(500,10_500,num=21)
denslv=np.linspace(0.2,4.0,num=20)

#scale_fac = 1.0 #1e-4 * 1.944**2 
#denslv= np.linspace(0.002,0.052,num=21) # - 0.05

nxplo=2
nyplo=1
xsize,ysize=0.7*20,0.7*15

fig = plt.figure(figsize=(nxplo*xsize, nyplo*ysize))
levs=[29,26,20,10]


n=1
Axes1 = Pu.axes_def(n=n,nxplo=nxplo,nyplo=nyplo, ypan=0.8 ) 
ax1 = fig.add_axes( Axes1 , projection=projPC )

ax1.set_global()
ax1.coastlines(resolution='110m',color='black')
#ax1.set_title( "Spectrally-nudged CAM : Sept 15-30 2017" , fontsize=16)
ax1.set_title( f"{camexp}" , fontsize=16)
ax1.add_feature(cfeature.LAND, facecolor='tan') 


ist=0
categories =[-2,0,1,2,3,4,5]  # -2 (sub-category storms) to 7 (Cat5)
cat_labels=['TD','TS','Cat 1','Cat 2','Cat 3','Cat 4','Cat 5',]
colors = ['beige','khaki','gold','orange', 'darkorange','red','red']  # Red for Cat5+
colors = ['green','green','gold','orange', 'darkorange','red','red']  # Red for Cat5+
colors = ['honeydew','green','gold','orange', 'darkorange','red','darkred']  # Red for Cat5+

nstorm,nt=np.shape( pdx.wind)
for ist in np.arange(nstorm):
    coo  = trx.categorize_storms( 0.9*pdx.wind[ist, :], wind=True) 

    # Plot each category with its color
    for cat, color, catlab in zip(categories, colors,cat_labels):
        #mask = ((coo == cat) & (ibx.day[ist,:]>15)& (ibx.month[ist,:]==9))
        #mask = ((coo == cat) & (ibx.month[ist,:]>=8) & (ibx.month[ist,:]<=9) )
        #mask = ((coo == cat)) #& (ibx.month[ist,:]>=8) & (ibx.month[ist,:]<=9) )
        #mask = ((coo == cat) & (pdx.basin[ist,:]==6)  ) #& (ibx.month[ist,:]>=8) & (ibx.month[ist,:]<=9) )
        mask = ((coo == cat) ) #& (pdx.basin[ist,:]==6)  ) #& (ibx.month[ist,:]>=8) & (ibx.month[ist,:]<=9) )
        ax1.scatter(pdx.lon[ist, mask], pdx.lat[ist, mask],
                    transform=data_crs, s=1., color=color , label=f"{catlab}" if ist == 0 else "")


"""
for ist in np.arange(len(tcMAAG)):
    windtc = np.array( tcMAAG[ist]['wind'] )
    lontc  = np.array( tcMAAG[ist]['lon'] )
    lattc  = np.array( tcMAAG[ist]['lat'] )
    coo  = trx.categorize_storms(  0.9*windtc , wind=True) 
    #ax1.scatter( ibx.lon[ist,:],ibx.lat[ist,:],transform=data_crs, s=.2 )


    # Plot each category with its color
    for cat, color, catlab in zip(categories, colors,cat_labels):
        mask = (coo == cat)
        ax1.scatter( lontc[mask]  , lattc[mask]  ,
                    transform=data_crs, s=5., color=color, label=f"{catlab}" if ist == 0 else "")

    #ax1.scatter( lontc  , lattc , transform=data_crs )
"""

# Focus on Caribbean, for example
west, east, south, north = -120, -1, 5, 55
#ax1.set_extent([west, east, south, north], crs=ccrs.PlateCarree())

handles, labels = ax1.get_legend_handles_labels()
legend=ax1.legend(
    handles, 
    labels, 
    handler_map={plt.scatter: HandlerPathCollection()},
    loc='upper left', 
    title="Categories", 
    fontsize=14,  # Text size
    title_fontsize=16  # Title size
)
#for handle in legend.legendHandles:
for handle in legend.legend_handles:
    handle.set_sizes([100])  # Adjust this value for desired marker size







########################################################################
n=2
Axes1 = Pu.axes_def(n=n,nxplo=nxplo,nyplo=nyplo, ypan=0.8 ) 
ax1 = fig.add_axes( Axes1 , projection=projPC )

ax1.set_global()
ax1.coastlines(resolution='110m',color='black')
#ax1.set_title( "IBTrACS Aug-Sep 2017" , fontsize=16)
#ax1.set_title( "IBTrACS :  Sept 15-30 2017" , fontsize=16)
ax1.set_title( "IBTrACS " , fontsize=16)
ax1.add_feature(cfeature.LAND, facecolor='tan') 


ist=0
categories =[-2,0,1,2,3,4,5]  # -2 (sub-category storms) to 7 (Cat5)
cat_labels=['TD','TS','Cat 1','Cat 2','Cat 3','Cat 4','Cat 5',]
colors = ['beige','khaki','gold','orange', 'darkorange','red','red']  # Red for Cat5+
colors = ['green','green','gold','orange', 'darkorange','red','red']  # Red for Cat5+
colors = ['honeydew','green','gold','orange', 'darkorange','red','darkred']  # Red for Cat5+



for ii in np.arange(len(yook[0])):
    ist=yook[0][ii]
    coo  = trx.categorize_storms(ibx.wind[ist, :], wind=True) 
    #ax1.scatter( ibx.lon[ist,:],ibx.lat[ist,:],transform=data_crs, s=.2 )

    # Plot each category with its color
    for cat, color, catlab in zip(categories, colors,cat_labels):
        #mask = ((coo == cat) & (ibx.day[ist,:]>15)& (ibx.month[ist,:]==9))
        #mask = ((coo == cat) & (ibx.month[ist,:]>=8) & (ibx.month[ist,:]<=9) )
        #mask = ((coo == cat)) #& (ibx.month[ist,:]>=8) & (ibx.month[ist,:]<=9) )
        #mask = ((coo == cat) & (ibx.basin[ist,:]==6)  ) #& (ibx.month[ist,:]>=8) & (ibx.month[ist,:]<=9) )
        mask = ((coo == cat) ) # & (ibx.basin[ist,:]==6)  ) #& (ibx.month[ist,:]>=8) & (ibx.month[ist,:]<=9) )
        ax1.scatter(ibx.lon[ist, mask], ibx.lat[ist, mask],
                    transform=data_crs, s=1., color=color , label=f"{catlab}" if ii == 0 else "")





# Focus on Caribbean, for example
west, east, south, north = -120, -1, 5, 55
#ax1.set_extent([west, east, south, north], crs=ccrs.PlateCarree())

handles, labels = ax1.get_legend_handles_labels()
legend=ax1.legend(
    handles, 
    labels, 
    handler_map={plt.scatter: HandlerPathCollection()},
    loc='upper left', 
    title="Categories", 
    fontsize=14,  # Text size
    title_fontsize=16  # Title size
)
#for handle in legend.legendHandles:
for handle in legend.legend_handles:
    handle.set_sizes([100])  # Adjust this value for desired marker size



In [ ]:
print(np.min(pdx.year))

In [ ]:
print(ibx.hour[2,0:5])
print(pdx.hour[2,0:5])
